# Homework 3
* CSCI-5930 ML Fall 2025


## Put details below:

In [ ]:
my_name = "Joseph Tesoriero"
my_collaborators = "  "

## Rules to work with this notebook
* You can create new cells in this notebook.
* PLEASE DO NOT DELETE ANY EXISTING CELL FROM THIS NOTEBOOK.
* In a cell, you are allowed only to write codes/snippets right after the marking like ##YOUR CODE. You are not allowed to write anywhere else in that particular cell.
* Your work for each task has to be in the specific cells with the ##YOUR CODE HERE marks, otherwise your submission won't be graded. Please do not delete the marks.
* After the ##YOUR CODE HERE marks, you may see command `raise NotImplementedError()` exception that alerts us that you did not attempt a particular task. Be sure to delete that command if you attempt to solve it.

In [ ]:
#Please make sure you are using a python3.9.x interpreter 
# and install the packages from requirements.txt (provided)
!pip install -r /kaggle/input/requirements-txt/requirements.txt

In [2]:
#Some basic imports. You can import any package at any cell
from sklearn.preprocessing import OneHotEncoder   #My favorite categorical to numerical feature conversion tool
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from numpy.linalg import inv
from time import perf_counter
import math
import sys
import random

In [3]:
### Let's load the given dataset
# First load the dataset into pandas dataframe
full_dataset = pd.read_csv('/kaggle/input/fall-25-birth-weight-prediction/baby-weights-dataset.csv',delimiter=',')
judge_dataset = pd.read_csv('/kaggle/input/fall-25-birth-weight-prediction/judge-without-labels.csv',delimiter=',')

## TASK 1: 
Separate the full_dataset into two parts: X and y, where X denotes the input matrix containing only the input (i.e., independent explanatory) variables, and y denotes the target variable containing only the target values for exactly the same number of samples in the given full_dataset. 

In [4]:
#Make sure you write your solution to this task below in this cell only.

# YOUR CODE HERE
X = full_dataset.iloc[:, :-1]
y = full_dataset.iloc[: , -1:].to_numpy().flatten()

In [5]:
#Let me test your code above
assert X.shape==(101400,36)
assert y.shape==(101400,)

## TASK 2:
* Given X representing the input matrix from the full_dataset, y being the target vector (the rightmost column of the full_dataset), obtained from Task 1: 
* randomly split the (X,y) dataset into 75% for training and 25% for testing using the library function from the library [sklearn.model_selection](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) . Please pass to the train_test_split function an additional argument random_state=45931.
* Store the 4 splits as X_train, X_test, y_train, y_test respectively.
* Save the ID column for X_train and X_test into ID_train and ID_test as list variable.
* Now, drop the ID columns from both X_train and X_test

In [95]:
#Make sure you write your solution to this task below in this cell only.

X_train = []
X_test = []
y_train = []
y_test = []

# YOUR CODE HERE
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 45931)
X_train = X_train.iloc[:, 1:]
X_test = X_test.iloc[:, 1:]

In [7]:
#Let me test your code above
assert (X_train.shape, X_test.shape, y_train.shape, y_test.shape) == ((76050, 35), (25350, 35), (76050,), (25350,))

## TASK 3:
Compute mean, stdev, min, max, 25% percentile, median and 75% percentile of BWEIGHT target variable (i.e, the target y) in the training set (i.e., y_train), and print the computed values as a numpy array containing these 7 results (respectively).


In [8]:
#Make sure you write your solution to this task below in this cell only.
mean_val = 0
stdev_val = 0
min_val = 0
max_val = 0
percentile25_val = 0
median_val = 0
percentile75_val = 0

# YOUR CODE HERE
mean_val = y_train.mean()
stdev_val = y_train.std()
min_val = y_train.min()
max_val = y_train.max()
percentile25_val = np.percentile(y_train, 25)
median_val = np.median(y_train)
percentile75_val = np.percentile(y_train, 75)

metrics = np.array([mean_val, stdev_val, min_val, max_val, percentile25_val, median_val, percentile75_val])
metrics

array([ 7.25699786,  1.33004969,  0.3125    , 13.0625    ,  6.625     ,
        7.375     ,  8.0625    ])

In [9]:
assert abs(sum([mean_val,stdev_val, min_val,max_val, percentile25_val,median_val,percentile75_val])-44.024547552217015) < 1e-4

## TASK 4: 

A little background first: Categorical features are features that contain values that are not numeric. As you can imagine non-numeric values will create trouble (by introducing `nan`) during calculation to gradients, and whatnot, right? The maths are undefined when these get in its way. An obvious solution you may be intrigued to do is dropping the features! Aha! Wrong!! Every piece of data is precious... as those non-numeric features may present with valuable insights of the data samples to find the patterns to map inputs with output/targets. So, we should include them. But, how?

The answer is via "Encoding" we can make good use the non-numeric variables.

There are several types of encoding used in practice. Here are the two popular ones:

1. **Label Encoding**, where labels are encoded as subsequent numbers. Say, for a categorical feature named "Category" with three categorical values: {“Cat”, “Dog” or “Zebra”} can be encoded to "0", "1", "2" respectively as in figure below. The issue with this type of encoding may unintentionally impose a type of ordering of the categories, that may add bias to the training.
![label-encoding](figs/le.png)
1. **One Hot Encoding**, ignores the ordering of the categories all together. With one-hot, we convert each categorical value into a new categorical column and assign a binary value of 1 or 0 to those columns. Each integer value is represented as a binary vector. All the values are zero, and the index is marked with a 1. Also, don't forget to remove the original categorical features. Here below just an example, how to convert the categorical feature called "Category" having the {“Cat”, “Dog” or “Zebra”} values into three new binary features: "Cat", "Dog", "Zebra".
![label-encoding](figs/ohe.png)

**A note on the Dummy Variable Trap**
The Dummy Variable Trap occurs when two or more dummy variables created by one-hot encoding are highly correlated (i.e., becomes multi-collinear). This means that one variable can be predicted from the others, making it difficult to interpret predicted coefficient variables in regression models. In other words, the individual effect of the dummy variables on the prediction model can not be interpreted well because of multicollinearity.

Using the one-hot encoding method, a new dummy variable is created for each categorical variable to represent the presence (1) or absence (0) of the categorical variable. For example, if tree species is a categorical variable made up of the values pine, or oak, then tree species can be represented as a dummy variable by converting each variable to a one-hot vector. This means that a separate column is obtained for each category, where the first column represents if the tree is pine and the second column represents if the tree is oak. Each column will contain a 0 or 1 if the tree in question is of the column's species. These two columns are multi-collinear since if a tree is pine, then we know it's not oak and vice versa. The machine learning models trained on dataset having this multi-collinearity suffers. A remedy is to drop first (or any one) of the dummy (i.e., one-hot) features created.

Given the training dataset (X_train, y_train), save as X_train_ohe after replacing all the non-numeric variables (i.e., categorical variables) with numeric encoding. Also, you need to use the same encoding to work on the X_train dataset as well and save it as X_test_oh. We need to make sure any new categorical value in the test dataset get ignored, i.e, when an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. Please complete the following `almost complete` function `lets_do_one_hot_encoding()` that encodes dataset on the supplied categorical feature column list.


In [10]:
#Make sure you write your solution to this task below in the marked location in this cell only.


def lets_do_one_hot_encoding(data, categorical_features, transform_only=True, encoders=[], verbose=False):
    '''
    The function does one_hot_encoding on the given dataset... 
    It's intentionally defined fully, meaning you do not need to do anything here... but show us
    how to use it properly.
    
    Input: 
        * data -- it's pretty much either X_train, or X_test that you prepared previously, that is
                  any dataframe having independent variables.
        * categorical_features -- a list of column/feature names in the data (dataframe) that you think
                  are non-numerical / i.e., categorical that you want to be converted to numerical
                  using the one-hot encoding technique.
        * transform_only -- a boolean parameter, if set to False, will learn (i.e., fit) various categorical
                values in the categorical_features from the given dataset, and use this to encode the dataset
                using one-hot encoding. This is important that you set to False on training dataset, and
                True on test set. If new categorical values are present in the test dataset, those will be
                ignored, making it easier to have same set of encoded features both in training and test 
                dataset... otherwise, subsequent operations (may/) will not work.
        * encoders -- a list of one-hot encoders previously saved, and now will be used to encode given dataset.
                If transform_only=True, the function looks for this provided list of encoders to encode the 
                dataset instead of learning new categories. Once again, it's expected that you pass the set
                of encoders for each of the categorical features that you encoded the training set -- i.e.,
                leave it empty for training set, and pass the set of encoders while encoding test set.
    Returns:
        * data -- the converted dataframe after the encoding is completed.
        * enc_list -- is the list of encoders used to encode the given data.
        * categorical_features -- is the list of categorical features that you would want to encode.
        It's expected that for training dataset, you save the encoder list and the list of features for later
        use to encode a test dataset.
    '''
    if len(encoders)>0:
        enc_list=encoders
    else:
        enc_list = []
    col_onehot = []
    i = 0
    for feature in categorical_features:
        if verbose: print('Dealing with feature ={}'.format(feature))
        if transform_only==True: #for test dataset
            enc = enc_list[i]
            c_onehot = enc.transform(data[[feature]])
            i = i + 1
        else: #fit and transform
            enc = OneHotEncoder(handle_unknown='ignore', drop='first', sparse=False)
            c_onehot = enc.fit_transform(data[[feature]])
            if verbose: print('c_onehot = {}'.format(c_onehot))
            enc_list.append(enc)
        c_onehot = pd.DataFrame(c_onehot, columns=list(enc.categories_[0][1:])) #dropped first column
        if verbose: print('dropped first column...c_onehot = {}'.format(c_onehot))
        c_onehot = c_onehot.add_prefix(feature+'_')
        col_onehot.append(c_onehot)
        if verbose: print('col_onehot = {}'.format(col_onehot))
    
    #concat all onehot feature columns
    concat_df = pd.concat(col_onehot,axis=1)
    #match index with given data
    concat_df.index = data.index
    
    #Here below is your task:
    #i) drop the categorical feature columns from dataframe `data`
    #ii) and then merge with those new onehot features stored in `cocat_df`
    # YOUR CODE HERE
    for feature in categorical_features:
        data = data.drop(feature, axis=1)

    data = pd.concat([data, concat_df], axis=1)
    
    return data,enc_list,categorical_features

In [120]:
'''
let's apply one-hot encoding on two columns: "HISPMOM" and "HISPDAD" of the training dataset, X_train.
Please call the function, lets_do_one_hot_encoding() you helped defining above passing appropriate arguments.
Also, receive the 3 return values from the function call: X_train_ohe, enc_list and categorical_features to 
use them in the next question, that is, encoding the test dataset, X_test, and 
later in Task 17 to encode the judge set. 

It's always a good idea to save the list of encoders and features in a file (e.g., joblib), if you would 
want to apply your model to evaluate/predict on a new test sample. In a summary: if your model was trained with
transformed one-hot encoded dataset, a new test data also needs to be encoded with the same list of variables
defined by the encoding.

'''
X_train_ohe = []  #one-hot encoded training dataset
enc_list = [] #encoder list to be created during one-hot encoding of the training dataset
categorical_features = ['HISPMOM', 'HISPDAD'] #The list of categorical features in question.

#Here below is your task:one-hot encoding of train set: X_train
# YOUR CODE HERE
X_train_ohe, enc_list, categorical_features = lets_do_one_hot_encoding(X_train, categorical_features, transform_only=False)

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [102]:
'''
let's apply one-hot encoding on the same two columns: "HISPMOM" and "HISPDAD" of the test dataset, X_test.
Please call the function, lets_do_one_hot_encoding() you helped defining above with appropriate parameters.
Be sure to use the encoders from the training set and turn on the `transform_only` switch to true.
Also, receive the 3 return values from the function call in X_test_ohe, enc_list, categorical_features. 
Although, the second and third returned values can be ignored.

Please make sure, the one-hot encoded training set and test set has the exact same number of features/columns,
and in the same order. If not, the test in the next cell will almost certainly fail, and you will lose points,
and the training and testing of the machine learning model will absolutely fail. So, please take close
attention.

'''
X_test_ohe = []  #one-hot encoded test dataset

#Here below is your task: one-hot encoding of test set: X_test
# YOUR CODE HERE
X_test_ohe, enc_list, categorical_features = lets_do_one_hot_encoding(X_test, categorical_features, transform_only=True, encoders=enc_list)

In [74]:
assert len(X_train_ohe.columns)== len(X_test_ohe.columns) and len(X_train_ohe.columns) == sum([1 for i, j in zip(X_train_ohe.columns, X_test_ohe.columns) if i == j])
assert (X_train_ohe.shape, X_test_ohe.shape)==((76050, 45), (25350, 45))

## TASK 5: 
* Given the X_train_ohe (Onehot encoded Pandas Dataframe from Task 4), check if there are missing values, and if yes, count how many, and impute the missing values with corresponding mean values. 
* Finally, print the counting result as a Pandas dataframe named "missing_counts" having 2 columns {variable_name,num_of_missing_values).  Please make sure that the result lists all the input variables in the given dataset. 
* Now, impute the missing values by mean of the respective variable and save the revised dataframe as X_train_ohe_imputed.

In [163]:
column_names = ["variable_name", "num_of_missing_values"]
missing_counts = pd.DataFrame()
X_train_ohe_imputed = X_train_ohe

#Your task below. 
# i) Please prepare the missing_counts dataframe accordingly.
# ii) impute missing value in a variable with corresponding mean of the variable.
# YOUR CODE HERE
missing_counts = pd.DataFrame(columns = column_names)
total = 0
index = []
X_train_ohe_imputed = X_train_ohe_imputed.copy()

for column_name in X_train_ohe.columns:
    column = X_train_ohe[column_name]
    missing = column.isnull().sum()
    total +=missing
    column_missing = column.isnull()
    missing_counts.loc[len(missing_counts)] = [column_name, missing]
    indices = X_train_ohe.loc[column_missing].index
    mean_val = column.mean()
    for i in indices:
        X_train_ohe_imputed.loc[i, column_name] = mean_val

print(total)

5


In [15]:
assert sum(missing_counts['num_of_missing_values'])==5
assert np.isnan(X_train_ohe.loc[[1748]][['FEDUC']].iloc[0,0])==True
assert np.isnan(X_train_ohe_imputed.loc[[1748]][['FEDUC']].iloc[0,0])==False

## TASK 6: 
* Given a X_train_ohe_imputed (Pandas dataframe from Task 5) where all the categorical variables are already replaced with numeric values, print a list of top 20 highly correlated variables with respect to the target variable, and save the result as a Pandas dataframe named top20_df with 2 columns {variable,corr_score}. 
* Here, the corr_score between a variable x and the target variable y needs to be computed using the Pearson Correlation Coefficient (PCC). Please note, PCC ranges between -1 to +1. PCC score 0 means no correlation, while value towards +1 and -1 represent positive and negative correlations respectively. For instance, PCC=0.8 and PCC=-0.8 tell similar strength positive and negative correlations between the two subject variables.
* Please do not include BWEIGHT in the top20_df list of top 20 correlated variable list.

In [16]:
top20_df = pd.DataFrame() #You need to save the top-20 most correlated variables with respect to BWEIGHT target

# Your task below is:
# i) on the imputed dataset you got from task 5, find the top-20 most-correlated variables
# with respect to the target variable.

# YOUR CODE HERE
y_train_series = pd.Series(y_train, name="target")
corr_scores = X_train_ohe_imputed.apply(lambda col: col.corr(y_train_series))
abs_val = corr_scores.abs()
top20 = abs_val.sort_values(ascending=False).head(20)
top20_df = pd.DataFrame({
    'variable': top20.index,
    'corr_score': top20.values
})
print(top20_df)

     variable  corr_score
0   HISPDAD_U    0.010076
1         SEX    0.008945
2     PINFANT    0.008020
3     RACEMOM    0.007004
4      VISITS    0.006080
5     RACEDAD    0.005793
6     HYPERCH    0.005769
7   HISPMOM_M    0.004862
8       WEEKS    0.004730
9   HISPMOM_N    0.004039
10     CERVIX    0.003839
11     HYDRAM    0.003534
12     HERPES    0.003488
13      FEDUC    0.003388
14      BDEAD    0.003333
15       FAGE    0.003058
16      MEDUC    0.002977
17  HISPMOM_P    0.002937
18   DIABETES    0.002830
19     GAINED    0.002771


In [17]:
assert top20_df.shape==(20,2)

## TASK 7: 
Given the X_train_ohe_imputed (as Pandas dataframe from task 5) and and top20_df (as Pandas Dataframe from Task 6) having 2 columns {variable_name,corr_score} similar to the one you computed in Task 6:
* Please save as X_train_t20 keeping only the columns listed in the top20_df dataframe.
* Repeat the process for X_test_ohe (obtained from task 4), and save it as X_test_t20.

In [18]:
#Your task below is to slice training and test dataset to retain only the top-20 most-correlated variables
# with respect to the target variable

X_train_t20 = pd.DataFrame()
X_test_t20 = pd.DataFrame()

# YOUR CODE HERE
columns = []
for i in range(len(top20_df)):
    columns.append(top20_df['variable'][i])

X_train_t20 = X_train_ohe_imputed[columns]
X_test_t20 = X_test_ohe[columns]

In [19]:
assert (X_train_t20.shape,X_test_t20.shape)==((76050, 20), (25350, 20))
assert len(X_train_t20.columns)== len(X_test_t20.columns) and len(X_train_t20.columns) == sum([1 for i, j in zip(X_train_t20.columns, X_test_t20.columns) if i == j])

## TASK 8: 
* Apply min-max scaling on the training dataset (X_train_t20 obtained from Task 7). Save the result as X_train_scaled_mm.
* Then scale the test dataset (X_test_t20 obtained from Task 7) based on the metrics you obtain when you scale the training dataset. Save the result as X_test_scaled_mm.
* PLEASE DO NOT SCALE y_train and y_test.


In [20]:
X_train_scaled_mm = np.array(X_train_t20)
X_test_scaled_mm = np.array(X_test_t20)

#Your task below:
# YOUR CODE HERE
scaler = MinMaxScaler()
X_train_scaled_mm = scaler.fit_transform(X_train_t20)
X_test_scaled_mm = scaler.fit_transform(X_test_t20)

In [21]:
assert abs(sum(np.array([min(X_train_scaled_mm[:,0]),max(X_train_scaled_mm[:,0]),min(X_test_scaled_mm[:,0]),max(X_test_scaled_mm[:,0])])-np.array([0.0,1.0,0.0,1.0])))<1e-4


## TASK 9: 
* Apply standardization (i.e., normalization) scaling on the training dataset (X_train_t20 obtained from Task 7). Save the result as X_train_scaled_std.
* Then scale the test dataset (X_test_t20 obtained from Task 7) based on the metrics you obtain when you scale the training dataset. Save the result as X_test_scaled_std.
* PLEASE DO NOT SCALE y_train and y_test.




In [22]:
X_train_scaled_std = np.array(X_train_t20)
X_test_scaled_std = np.array(X_test_t20)

#Your task below:
# YOUR CODE HERE
scaler = StandardScaler()
X_train_scaled_std = scaler.fit_transform(X_train_t20)
X_test_scaled_std = scaler.fit_transform(X_test_t20)


In [23]:
for i in np.arange(20):
    assert abs(X_train_scaled_std[:,i].mean()-0.0)<1e-4 and abs(X_train_scaled_std[:,i].std()-1.0)<1e-4


## TASK 10: 
Given the (X_train_scaled_std, y_train) pairs denoting input matrix and output vector respectively: complete the three function definitions and demonstrate the functionalities of each by calling them with appropriate arguments as instructed below:
* **linear_regression_closed_form_training** : It fits a linear regression model using the closed-form solution to obtain the coefficients, beta's, as a numpy array of m+1 values (Please recall class lecture), where *m* is the number of variables kept in X_train (the first argument to the function). Please measure the cpu_time needed during the training step. cpu_time is not equal to the wall_time. So, use time.perf_counter() for an accurate measurement. Documentation on this function can be found here: https://docs.python.org/3/library/time.html . Finally, the function returns betas (i.e., the m+1 beta values) and the  cpu_time.
* **linear_regression_closed_form_predict**: It takes a list of m+1 beta values (i.e., betas returned from the corresponding training function, and X_test (containing test samples each having *m* input variables). Now, using the provided beta values, predict each of the test samples provided, and let's name your prediction "y_pred". Return y_pred from the function.
* **RMSLE**: It takes two lists: y_test, y_pred, where the first list represents ground truth (i.e., actual) target values for the given samples, and the second list represents a corresponding predicted values for exactly same number of samples in y_test. Compute and return the Root Mean Squared Logarithmic Error (RMSLE) of the prediction. 

* PLEASE DO NOT USE ANY LIBRARY FUNCTION THAT DOES THE LINEAR REGRESSION or RMSLE calculation.
* Now, call linear_regression_closed_form_training() function providing X_train_scaled_std, y_train obtained from Task 9, and save the returned results as betas_closed_form,cpu_time_closed_form.
* Print betas_closed_form, cpu_time_closed_form
* Call linear_regression_closed_form_predict() function providing betas_closed_form,X_test_scaled_std obtained in Task 9. Save the returned result as y_pred.
* Call RMSLE() function providing y_test and y_pred. Save returned result as rmsle_closed_form.
* Print rmsle_closed_form.

In [24]:
import time

def linear_regression_closed_form_training(X_train, y_train):
    betas = []
    cpu_time = 0

    #Your work here
    # YOUR CODE HERE
    start = time.perf_counter()
    X_train_scaled_array = np.array(X_train)
    X_train_betas = np.c_[np.ones((X_train_scaled_array.shape[0], 1)), X_train_scaled_array]
    betas = np.linalg.pinv(X_train_betas.T @ X_train_betas) @ X_train_betas.T @ y_train
    end = time.perf_counter()

    cpu_time = end-start
    
    return betas, cpu_time

def linear_regression_closed_form_predict(betas, X_test):
    y_pred = []
    #your work below
    # YOUR CODE HERE
    X_test_betas = np.c_[np.ones((X_test.shape[0], 1)), X_test]
    y_pred_array = X_test_betas @ betas
    y_pred = y_pred_array.tolist()
    return y_pred

def RMSLE(y_test, y_pred, verbose=False):
    rmsle_val = 0
    #Your work below
    # YOUR CODE HERE
    y_pred = np.array(y_pred)
    y_pred = np.maximum(y_pred, 0)
    rmsle = np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_test))**2))
    return rmsle


In [25]:
betas_closed_form = []
cpu_time_closed_form = 0
#Your task here, call appropriate function to get these two
# YOUR CODE HERE
betas_closed_form, cpu_time_closed_form = linear_regression_closed_form_training(X_train_scaled_std, y_train)
betas_closed_form = np.array(betas_closed_form)

In [26]:
y_pred = []
#Your task here. call appropriate function to get get y prediction for the supplied test dataset scaled
# YOUR CODE HERE
y_pred = np.array(linear_regression_closed_form_predict(betas_closed_form, X_test_scaled_std))

In [27]:
rmsle_closed_form = 0
#Your task below. Call the function to compute RMSLE score of the y predictions
# YOUR CODE HERE
rmsle_closed_form = RMSLE(y_test, y_pred)

In [28]:
assert abs(RMSLE([1,2,3,4,5],[1,1,1,1,1])-0.733674673672524)<1e-4
assert abs(RMSLE([100, 200, 300, 400, 500], [90, 190, 290, 390, 490])-0.05596497907273607)<1e-4
assert (betas_closed_form.shape[0],y_pred.shape[0])==(21,25350)

## TASK 11: 
Given the (X_train_scaled_std, y_train) pairs denoting input matrix and output vector respectively: complete the three function definitions and demonstrate the functionalities of each by calling them with appropriate arguments as instructed below:
* **linear_regression_gd_batch_training** : It fits a linear regression model using the batch gradient descent algorithm to obtain the coefficients, beta's, as a numpy array of m+1 values, where *m* is the number of variables kept in X_train (the first argument to the function). Make sure you compute average of gradients in the batch. Please use the alpha (i.e, the learning rate) and nEpoch (number of epochs) parameters in your implementation of the gradient descent algorithm. Please measure the cpu_time needed during the training step. cpu_time is not equal to the wall_time. So, use time.perf_counter() for an accurate measurement. Documentation on this function can be found here: https://docs.python.org/3/library/time.html . Finally, the function returns betas (i.e., the m+1 beta values) and the  cpu_time.
* **linear_regression_gd_batch_predict**: It takes a list of m+1 beta values (i.e., betas returned from the corresponding training function, and X_test (containing test samples each having *m* input variables). Now, using the provided beta values, predict each of the test samples provided, and let's name your prediction "y_pred". Return y_pred from the function. 

* PLEASE DO NOT USE ANY LIBRARY FUNCTION THAT DOES THE LINEAR REGRESSION.
* Now, call linear_regression_gd_batch_training() function providing X_train_scaled_std, y_train obtained from Task 9, and alpha=0.01,nEpoch=1000, and save the returned results as betas_batch,cpu_time_batch.
* Print betas_batch, cpu_time_batch
* Call linear_regression_gd_batch_predict() function providing betas_batch,X_test_scaled_std obtained in Task 9. Save the returned result as y_pred.
* Call RMSLE() function providing y_test and y_pred. Save returned result as rmsle_batch.
* Print rmsle_batch.



In [29]:
def linear_regression_gd_batch_training(X_train,y_train, alpha, nEpoch):
    random.seed(554433)
    betas = []
    cpu_time = 0
    epsilon = 1e-6
    #Your task below
    # YOUR CODE HERE
    start = time.perf_counter()
    X_train_scaled_array = np.array(X_train)
    X_train_betas = np.c_[np.ones((X_train_scaled_array.shape[0], 1)), X_train_scaled_array]
    X_train_betas = X_train_betas.astype(np.float32)
    y_train = y_train.reshape(-1, 1)
    y_train = y_train.astype(np.float32)
    n, m = X_train_betas.shape
    betas = np.zeros((m, 1))
    
    for i in range(nEpoch):
        y_pred = X_train_betas @ betas
        gradient = (1/n)*X_train_betas.T @ (y_pred - y_train)
        if np.linalg.norm(gradient) < epsilon: # convergence check
            break
        betas = betas - alpha*gradient

    end = time.perf_counter()
    cpu_time = end - start
        
    return betas, cpu_time

def linear_regression_gd_batch_predict(betas,X_test):
    y_pred = []
    
    #Your task below
    # YOUR CODE HERE
    X_test_betas = np.c_[np.ones((X_test.shape[0],1)), X_test]
    y_pred = X_test_betas @ betas
    
    return y_pred

In [30]:
betas_batch = []
cpu_time_batch = 0

#your task below
# YOUR CODE HERE
betas_batch, cpu_time_batch = linear_regression_gd_batch_training(X_train_scaled_std, y_train, 0.01, 1000)
betas_batch = np.array(betas_batch)

In [31]:
y_pred = []

#your task below
# YOUR CODE HERE
y_pred = linear_regression_gd_batch_predict(betas_batch, X_test_scaled_std)
print(y_pred)

[[7.35581937]
 [6.68156158]
 [7.0335397 ]
 ...
 [7.8130925 ]
 [6.53834659]
 [7.33270273]]


In [32]:
rmsle_batch = 0
# YOUR CODE HERE
rmsle_batch = RMSLE(y_test, y_pred)


## TASK 12:
Given the (X_train_scaled_std, y_train) pairs denoting input matrix and output vector respectively: complete the three function definitions and demonstrate the functionalities of each by calling them with appropriate arguments as instructed below:
* **linear_regression_gd_stochastic_training** : It fits a linear regression model using the stochastic gradient descent algorithm to obtain the coefficients, beta's, as a numpy array of m+1 values, where *m* is the number of variables kept in X_train (the first argument to the function). Please use the alpha (i.e, the learning rate), nEpoch (number of epochs), nIteration (number of iterations) parameters in your implementation of the gradient descent algorithm. Please measure the cpu_time needed during the training step. cpu_time is not equal to the wall_time. So, use time.perf_counter() for an accurate measurement. Documentation on this function can be found here: https://docs.python.org/3/library/time.html . Finally, the function returns betas (i.e., the m+1 beta values) and the  cpu_time.
* **linear_regression_gd_stochastic_predict**: It takes a list of m+1 beta values (i.e., betas returned from the corresponding training function, and X_test (containing test samples each having *m* input variables). Now, using the provided beta values, predict each of the test samples provided, and let's name your prediction "y_pred". Return y_pred from the function. 

* PLEASE DO NOT USE ANY LIBRARY FUNCTION THAT DOES THE LINEAR REGRESSION.
* Now, call linear_regression_gd_stochastic_training() function providing X_train_scaled_std, y_train obtained from Task 9, and alpha=0.001,nEpoch=100, nIteration=10000 , and save the returned results as betas_stochastic,cpu_time_stochastic.
* Print betas_stochastic, cpu_time_stochastic
* Call linear_regression_gd_stochastic_predict() function providing betas_stochastic,X_test_scaled_std obtained in Task 9. Save the returned result as y_pred.
* Call RMSLE() function providing y_test and y_pred. Save returned result as rmsle_stochastic.
* Print rmsle_stochastic.


In [33]:
def linear_regression_gd_stochastic_training(X_train,y_train,alpha,nEpoch,nIteration):
    random.seed(554433)
    betas = []
    cpu_time = 0
    epsilon = 1e-6

    ## YOUR CODE HERE
    # YOUR CODE HERE
    start = time.perf_counter()
    X_train_scaled_array = np.array(X_train)
    X_train_betas = np.c_[np.ones((X_train_scaled_array.shape[0], 1)), X_train_scaled_array]
    X_train_betas = X_train_betas.astype(np.float32)
    y_train = y_train.reshape(-1, 1)
    y_train = y_train.astype(np.float32)
    n, m = X_train_betas.shape
    betas = np.zeros((m, 1))

    iterations = 0
    
    for i in range(nEpoch):
        #Shuffle data
        indices = np.arange(n)
        np.random.shuffle(indices)
        X_shuffle = X_train_betas[indices]
        y_shuffle = y_train[indices]
        
        for j in range(n):
            
            if iterations >= nIteration: # Reached max iterations
                break
            # Loop over all samples
            x = X_shuffle[j].reshape(1, -1)
            y = y_shuffle[j]
            y_pred = x @ betas
            gradient = x.T @ (y_pred - y)

            if np.linalg.norm(gradient) < epsilon: # convergence check
                break
            
            betas = betas - alpha * gradient
            iterations += 1

        if np.linalg.norm(gradient) < epsilon: # convergence check
            break 

        if iterations >= nIteration: # Reached max iterations
            break
            
    end = time.perf_counter()
    cpu_time = end - start
    return betas, cpu_time

def linear_regression_gd_stochastic_predict(betas, X_test):
    y_pred = []
    
    #your code below
    # YOUR CODE HERE
    X_test_betas = np.c_[np.ones((X_test.shape[0],1)), X_test]
    y_pred = X_test_betas @ betas
    return y_pred

In [34]:
betas_stochastic = []
cpu_time_stochastic = 0
#your code below
# YOUR CODE HERE
betas_stochastic, cpu_time_stochastic = linear_regression_gd_stochastic_training(X_train_scaled_std, y_train, 0.001, 100, 10000)
print(betas_stochastic)

[[ 7.26476753e+00]
 [-3.18136712e-02]
 [-9.04373862e-02]
 [ 9.67320042e-02]
 [-2.40431141e-02]
 [ 3.95043440e-02]
 [-6.97215077e-02]
 [-3.78602742e-02]
 [ 7.09418985e-02]
 [ 7.47027787e-01]
 [ 5.14641360e-03]
 [-2.16625226e-02]
 [-6.67391936e-02]
 [-1.74334164e-02]
 [ 2.03368624e-02]
 [-1.91613808e-02]
 [ 9.62855620e-02]
 [ 2.75129132e-02]
 [-7.33652610e-03]
 [ 5.43359336e-02]
 [ 2.11705792e-01]]


In [35]:
y_pred = []
#your code below
# YOUR CODE HERE
y_pred = linear_regression_gd_stochastic_predict(betas_stochastic, X_test_scaled_std)
print(y_pred)

[[7.3942791 ]
 [6.58138802]
 [7.01861312]
 ...
 [7.894309  ]
 [6.54197956]
 [7.28661505]]


In [36]:
rmsle_stochastic = 0
#your code below
# YOUR CODE HERE
rmsle_stochastic = RMSLE(y_test, y_pred)

## Task 13: 
Given the (X_train_scaled_std, y_train) pairs denoting input matrix and output vector respectively: complete the three function definitions and demonstrate the functionalities of each by calling them with appropriate arguments as instructed below:
* **linear_regression_gd_minibatch_training** : It fits a linear regression model using the minibatch gradient descent algorithm to obtain the coefficients, beta's, as a numpy array of m+1 values, where *m* is the number of variables kept in X_train (the first argument to the function). Please use the alpha (i.e, the learning rate), nEpoch (number of epochs), nIteration (number of iterations), and batch_size parameters in your implementation of the gradient descent algorithm. Please measure the cpu_time needed during the training step. cpu_time is not equal to the wall_time. So, use time.perf_counter() for an accurate measurement. Documentation on this function can be found here: https://docs.python.org/3/library/time.html . Finally, the function returns betas (i.e., the m+1 beta values) and the  cpu_time.
* **linear_regression_gd_minibatch_predict**: It takes a list of m+1 beta values (i.e., betas returned from the corresponding training function, and X_test (containing test samples each having *m* input variables). Now, using the provided beta values, predict each of the test samples provided, and let's name your prediction "y_pred". Return y_pred from the function. 

* PLEASE DO NOT USE ANY LIBRARY FUNCTION THAT DOES THE LINEAR REGRESSION.
* Now, call linear_regression_gd_minibatch_training() function providing X_train_scaled_std, y_train obtained from Task 9, and alpha=0.01,nEpoch=50, nIteration=1000,batch_size=32, and save the returned results as betas_minibatch,cpu_time_minibatch.
* Print betas_minibatch, cpu_time_minibatch
* Call linear_regression_gd_minibatch_predict() function providing betas_minibatch,X_test_scaled_std obtained in Task 9. Save the returned result as y_pred.
* Call RMSLE() function providing y_test and y_pred. Save returned result as rmsle_minibatch.
* Print rmsle_minibatch.

In [38]:
def linear_regression_gd_minibatch_training(X_train,y_train,alpha,nEpoch, nIteration, batch_size):
    random.seed(554433)
    cpu_time = 0
    epsilon = 1e-6
    betas = []
    ## your code below
    # YOUR CODE HERE
    start = time.perf_counter()
    X_train_scaled_array = np.array(X_train)
    X_train_betas = np.c_[np.ones((X_train_scaled_array.shape[0], 1)), X_train_scaled_array]
    X_train_betas = X_train_betas.astype(np.float32)
    y_train = y_train.reshape(-1, 1)
    y_train = y_train.astype(np.float32)
    n, m = X_train_betas.shape
    betas = np.zeros((m, 1))
    number_batches = n // batch_size

    iterations = 0
    
    for i in range(nEpoch):
        # Shuffle data
        indices = np.arange(n)
        np.random.shuffle(indices)
        
        for j in range(0, n, batch_size):
            if iterations >= nIteration: # Reached max iterations
                break
            # Calculate batch
            batch = indices[j: j + batch_size]
            X_train_subset = X_train_betas[batch]
            y_train_subset = y_train[batch]
            y_pred = X_train_subset @ betas
            gradient = (1/batch_size)*X_train_subset.T @ (y_pred - y_train_subset)

            if np.linalg.norm(gradient) < epsilon: # convergence check
                break
            
            betas = betas - alpha*gradient

            iterations += 1

        if np.linalg.norm(gradient) < epsilon: # convergence check
            break
        
        if iterations >= nIteration: # Reached max iterations
            break

    end = time.perf_counter()
    cpu_time = end - start
    
    return betas,cpu_time

def linear_regression_gd_minibatch_predict(betas,X_test):
    #your solution below
    y_pred = []
    
    # YOUR CODE HERE
    X_test_betas = np.c_[np.ones((X_test.shape[0],1)), X_test]
    y_pred = X_test_betas @ betas
    return y_pred

In [39]:
betas_minibatch = []
cpu_time_minibatch = 0
#your task below

# YOUR CODE HERE
betas_minibatch, cpu_time_minibatch = linear_regression_gd_minibatch_training(X_train_scaled_std, y_train, 0.01, 50, 1000, 32)

In [40]:
y_pred = []
#your ans below
# YOUR CODE HERE
y_pred = linear_regression_gd_minibatch_predict(betas_minibatch, X_test_scaled_std)

In [41]:
rmsle_minibatch = 0
#your ans below
# YOUR CODE HERE
rmsle_minibatch = RMSLE(y_test, y_pred)

## Task 14:
Given the 4 sets of results from the 4 experiments (from Tasks 10, 11, 12, 13) with closed form solution, batch gradient descent, stochastic gradient descent and mini-batch gradient descent, assign a string from the set {"closed-form", "batch-GD", "stochastic-GD", "minibatch-GD"} that demonstrated the best predictive performance in terms of RMSE to a variable named `best_name`.


In [42]:
names = ["closed-form", "batch-GD", "stochastic-GD", "minibatch-GD"]
rmsle_vals = [rmsle_closed_form, rmsle_batch, rmsle_stochastic, rmsle_minibatch]

best_name = 'xxxx'
# YOUR CODE HERE
min_val = min(rmsle_vals)
best = rmsle_vals.index(min_val)
best_name = names[best]

for i in range(len(rmsle_vals)):
    print(f"{names[i]}: {rmsle_vals[i]}")
print(best_name)

closed-form: 0.14053410350974654
batch-GD: 0.2241158541214495
stochastic-GD: 0.2271218053574825
minibatch-GD: 0.22211921259125103
closed-form


## Task 15: 
Given the 4 sets of results from the 4 experiments (from Tasks 10, 11, 12, 13) with closed form solution, batch gradient descent, stochastic gradient descent and mini-batch gradient descent, assign a string from the set {"closed-form", "batch-GD", "stochastic-GD", "minibatch-GD"} that demonstrated the least training cpu time to a variable called `best_name`.


In [44]:
names = ["closed-form", "batch-GD", "stochastic-GD", "minibatch-GD"]
cpu_times = [cpu_time_closed_form, cpu_time_batch, cpu_time_stochastic, cpu_time_minibatch]
best_name = 'xxxx'

# YOUR CODE HERE
min_val = min(cpu_times)
best = cpu_times.index(min_val)
best_name = names[best]

for i in range(len(cpu_times)):
    print(f"{names[i]}: {cpu_times[i]}")
print(best_name)

closed-form: 0.09860456400002704
batch-GD: 6.747379609000006
stochastic-GD: 0.17087916599996333
minibatch-GD: 0.0639838229999441
minibatch-GD


## Task 16: 
Given the (X_train_scaled_std, y_train) pairs denoting input matrix and output vector respectively, 
* call your implementation of Task 13: minibatch gradient descent based linear regression for each of these learning rates: {0.001, 0.01, 0.05}, batch sizes: {32, 64, 128, 256}
    * Please use the nIteration (number of iterations)=100, nEpoch (number of epoch)=50.

* For each of the linear regression model, using the computed beta values, predict the test samples provided in the "X_test_scaled_std" argument, and let's name your prediction "y_pred".
* Compute Root Mean Squared Logarithmic Error (RMSLE) of your prediction using the RMSLE() function you defined in Task 10.
* Finally, assign the learning rate that shows the best test performance to a variable called `best_alpha`, and also assign as a pandas dataframe named `summary` with 3 columns: {learning_rate, batch_size, test_RMSLE} containing RMSLE's of the 12 linear regression models, i.e., Cartesian product between the alphas and batch_sizes. Also, assign the best performing batch_size to a variable `best_batch_size`.
* PLEASE DO NOT USE ANY LIBRARY FUNCTION THAT DOES THE LINEAR REGRESSION.


In [145]:
alphas = [0.001, 0.01, 0.05]
batch_sizes = [32, 64, 128, 256]
summary = pd.DataFrame(columns=['learning_rate', 'batch_size', 'test_RMSLE'])
nIteration=1000
nEpoch=50

best_alpha = -1
best_beta = []
best_batch_size = -1

#your code below
# YOUR CODE HERE
rmsle_values = np.zeros([3,4])

for i in range(len(alphas)):
    for j in range(len(batch_sizes)):
        betas, cpu = linear_regression_gd_minibatch_training(X_train_scaled_std, y_train, alphas[i], 50, 100, batch_sizes[j])
        y_pred = linear_regression_gd_minibatch_predict(betas, X_test_scaled_std)
        rmsle_value = RMSLE(y_test, y_pred)
        rmsle_values[i,j] = rmsle_value
        summary.loc[len(summary)] = [alphas[i], batch_sizes[j], rmsle_value]

min_index = summary['test_RMSLE'].idxmin()
best_alpha = summary.loc[min_index, 'learning_rate']
best_batch_size = summary.loc[min_index, 'batch_size']

print(best_alpha)
print(best_batch_size)

summary

0.05
128.0


,learning_rate,batch_size,test_RMSLE
0,0.001,32.0,1.588527
1,0.001,64.0,1.585242
2,0.001,128.0,1.585290
3,0.001,256.0,1.583945
4,0.010,32.0,0.441797
5,0.010,64.0,0.438603
6,0.010,128.0,0.438315
7,0.010,256.0,0.438687
8,0.050,32.0,0.226939
9,0.050,64.0,0.223579


## Task 17:
* Utilizing the best trained linear regression model (so far), predict the target for each of the samples in the judge_dataset.
    * I believe you will not forget to do the following before call in the prediction algorithm:
        - Save the ID values of the judge dataset into ID_judge and drop it from the judge dataframe.
        - Perform onehot encoding using the same encoder you used to encode X_test (Task 4). 
        - keep only the same top 20 variables as you did in Task 7. 
        - scale the input variables based on the same metrics you used to scale the training dataset (Task 9). 
    * Now, call the prediction function of that model to obtain y_pred.
    * Prepare a pandas dataframe called `result` having columns: {ID, BWEIGHT}, where ID will the ID of the judge sample, and BWEIGHT is the corresponding y_pred value from your model prediction.
    * Save the dataframe as `my_submission.csv`.
* PLEASE DO NOT USE ANY LIBRARY FUNCTION THAT DOES THE LINEAR REGRESSION.


In [141]:
#your task below
#i) store the id in a separate variable and drop it from judge dataset
#ii) use onehot encoder you obtained to encode the judge set
#iii) keep only the top-20 variables in the judge set
#iv) scale using the same scalar you obtained before, and save it to judge_scaled_std variable.
judge_scaled_std = []

judge_dataset = pd.read_csv('/kaggle/input/fall-25-birth-weight-prediction/judge-without-labels.csv',delimiter=',')

# Save and seperate IDs
judge_ID = judge_dataset.iloc[:, 0]
judge_dataset = judge_dataset.iloc[:, 1:]


# One hot encode data for training
categorical_features = ["HISPMOM", "HISPDAD"]
# Generate encoders
X_train_ohe, enc_list_judge, categorical_features = lets_do_one_hot_encoding(X_train, categorical_features, transform_only=False)
judge_ohe, enc_list_judge, categorical_features = lets_do_one_hot_encoding(judge_dataset, categorical_features, transform_only=True, encoders = enc_list_judge)

# Find any missing values and impute with the mean
total_judge = 0
judge_ohe_imputed = judge_ohe.copy()
for column_name in judge_ohe.columns:
    column = judge_ohe[column_name]
    missing = column.isnull().sum()
    total += missing
    column_missing = column.isnull()
    indices = judge_ohe.loc[column_missing].index
    mean_val = column.mean()
    for i in indices:
        judge_ohe_imputed.loc[i, column_name] = mean_val


# Scale Data
judge_t20 = pd.DataFrame()

judge_t20 = judge_ohe_imputed[columns]

judge_scaled_std = np.array(judge_t20)

scaler = StandardScaler()
judge_scaled_std = scaler.fit_transform(judge_t20)

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [156]:
# New cell to tune hyperparameters for full batch
alphas_full = [0.03, 0.04, 0.05, 0.055, 0.06]
epochs_full = [50, 100]
number_runs = 3

summary_full = pd.DataFrame(columns=['learning_rate', 'epochs', 'mean_RMSLE', 'std_RMSLE'])

for alpha in alphas_full:
    for epoch in epochs_full:
        rmsle_list = []
        for run in range(number_runs):
            betas, cpu_time = linear_regression_gd_batch_training(X_train_scaled_std, y_train, alpha, epoch)
            y_pred = linear_regression_gd_batch_predict(betas, X_test_scaled_std)
            rmsle_list.append(RMSLE(y_test, y_pred))
        
        summary_full.loc[len(summary_full)] = [alpha, nEpoch, np.mean(rmsle_list), np.std(rmsle_list)]

min_index_full = summary_full['mean_RMSLE'].idxmin()
best_alpha_full = summary_full.loc[min_index_full, 'learning_rate']
best_epoch_full = summary_full.loc[min_index_full, 'epochs']

print(f"Best alpha: {best_alpha_full}")
print(f"Best epochs: {best_epoch_full}")
summary_full

Best alpha: 0.05
Best epochs: 50.0


,learning_rate,epochs,mean_RMSLE,std_RMSLE
0,0.030,50.0,0.302404,0.0
1,0.030,50.0,0.226352,0.0
2,0.040,50.0,0.250374,0.0
3,0.040,50.0,0.223903,0.0
4,0.050,50.0,0.232018,0.0
5,0.050,50.0,0.223884,0.0
6,0.055,50.0,0.228191,0.0
7,0.055,50.0,0.223947,0.0
8,0.060,50.0,0.226059,0.0
9,0.060,50.0,0.224000,0.0


In [152]:
# New cell to tune hyperparameters for mini-batch

alphas_judge = [0.03, 0.04, 0.045, 0.05, 0.055, 0.06]
batch_sizes_judge = [96, 128, 160, 192]
number_runs = 5

summary_judge = pd.DataFrame(columns=['learning_rate', 'batch_size', 'mean_RMSLE', 'std_RMSLE'])

for alpha in alphas_judge:
    for batch in batch_sizes_judge:
        rmsle_list = []
        for run in range(number_runs):
            betas, cpu_time = linear_regression_gd_minibatch_training(X_train_scaled_std, y_train, alpha, nEpoch=50, nIteration=100, batch_size=batch)
            y_pred = linear_regression_gd_minibatch_predict(betas, X_test_scaled_std)
            rmsle_list.append(RMSLE(y_test, y_pred))
        
        summary_judge.loc[len(summary_judge)] = [alpha, batch, np.mean(rmsle_list), np.std(rmsle_list)]

min_index_judge = summary_judge['mean_RMSLE'].idxmin()
best_alpha_judge = summary_judge.loc[min_index_judge, 'learning_rate']
best_batch_size_judge = summary_judge.loc[min_index_judge, 'batch_size']

print(f"Best alpha: {best_alpha_judge}")
print(f"Best batch size: {best_batch_size_judge}")

summary_judge

Best alpha: 0.05
Best batch size: 192.0


,learning_rate,batch_size,mean_RMSLE,std_RMSLE
0,0.030,96.0,0.226634,0.000968
1,0.030,128.0,0.227175,0.001698
2,0.030,160.0,0.226548,0.000819
3,0.030,192.0,0.226569,0.001452
4,0.040,96.0,0.223922,0.000874
5,0.040,128.0,0.223836,0.000862
6,0.040,160.0,0.226038,0.001183
7,0.040,192.0,0.222883,0.001080
8,0.045,96.0,0.222838,0.000902
9,0.045,128.0,0.224003,0.002004


In [155]:
# After fine tuning the alpha and batch size, lets check different 
epochs_testing = [50, 100, 200, 400]
summary_epochs = pd.DataFrame(columns=['epochs', 'RMSLE'])

for epoch in epochs_testing:
    betas, cpu_time = linear_regression_gd_minibatch_training(X_train_scaled_std, y_train, alpha=0.05, nEpoch=epoch, nIteration=80000, batch_size=192)
    y_pred = linear_regression_gd_minibatch_predict(betas, X_test_scaled_std)
    rmsle_val = RMSLE(y_test, y_pred)
    summary_epochs.loc[len(summary_epochs)] = [epoch, rmsle_val]

summary_epochs

,epochs,RMSLE
0,50.0,0.223351
1,100.0,0.223476
2,200.0,0.225353
3,400.0,0.225368


In [161]:
#your task is to predict using your best regression betas, and save it to `y_pred` variable
y_pred = []

# YOUR CODE HERE

# Try predicting using closed form, since this returned the lowest RMSLE value for our initial testing
# y_pred = linear_regression_closed_form_predict(betas_closed_form, judge_scaled_std)

# Try predicting using mini-batch, using the best alpha and batch size from task 16
# betas_mini_judge, cpu_mini_judge = linear_regression_gd_minibatch_training(X_train_scaled_std, y_train, .05, 50, 100, 128)
# y_pred = linear_regression_gd_minibatch_predict(betas_mini_judge, judge_scaled_std)

# Using mini-batch again, but fine tuned the hyperparameters
betas_mini_judge_2, cpu_mini_judge_2 = linear_regression_gd_minibatch_training(X_train_scaled_std, y_train, .05, 50, 100, 192)
y_pred = linear_regression_gd_minibatch_predict(betas_mini_judge_2, judge_scaled_std)

# Full batch gradient, using alpha = .05 and epochs = 50
# betas_batch_judge, cpu_time_batch_judge = linear_regression_gd_batch_training(X_train_scaled_std, y_train, 0.05, 50)
# y_pred = linear_regression_gd_batch_predict(betas_batch_judge, judge_scaled_std)

# Last submission, changing alpha slightly to .045
# betas_mini_judge_3, cpu_mini_judge_3 = linear_regression_gd_minibatch_training(X_train_scaled_std, y_train, .045, 50, 100, 192)
# y_pred = linear_regression_gd_minibatch_predict(betas_mini_judge_3, judge_scaled_std)

In [162]:
#your task below is to prepare a dataframe `result` with columns: {ID,BWEIGHT} as defined in task-17
result = pd.DataFrame()
# YOUR CODE HERE
column_names_result = ['ID', 'BWEIGHT']
result = pd.DataFrame(columns = column_names_result)
result['ID'] = judge_ID
result['BWEIGHT'] = y_pred
result.to_csv('my_submission.csv',index=False)

## Task 18: 
* [Submit my_submission.csv at Kaggle url https://www.kaggle.com/competitions/fall-25-birth-weight-prediction ](https://www.kaggle.com/competitions/fall-25-birth-weight-prediction)
* [**Very important**] Please look back from tasks 1-17 to see either do hyperparameter tuning, data analysis, model selection to improve your initial submission.
* Please add details about your best submission by assigning appropriate values from your latest Kaggle submission:
    * kaggle_user_id -- your kaggle userid/handle
    * my_name -- my `actual` full name.
    * submission_date -- latest date/timestamp of your submission that you want to show/talk about here.
    * score -- the score you got at kaggle for that submission
    * entries -- number of entries you made/submitted in Kaggle for this task
    * tell_about_the_model -- a string explaining your choice of model, parameters, data analysis, etc.

In [164]:
kaggle_user_id = 'xxxxx'
my_name = 'xxxxx'
submission_date = 'xxxxx'
score = -1
entries = -1
tell_about_the_model = 'Here is how many epochs I ran, with learning rate blah blah..'

#your task is to re-assign proper values based on the task requirement below.
# YOUR CODE HERE
kaggle_user_id = 'josephtesoriero'
my_name = 'Joseph Tesoriero'
submission_date = '10/14/2025'
score = .2473
entries = 5
tell_about_the_model = "The model used was mini-batch gradient descent. I used an alpha = .05, batch size = 192, and number of epochs = 50. " \
                       "First I chose the alpha and batch size we determine in task 16. " \
                       "In order to fine tune further, I predicted on various alphas and batch sizes around the ideal values from task 16 " \
                       "I predicted the values for each alpha and batch size pair 5 time and computed the average and standard deviations " \
                       "The results from this test determine that alpha = .05 and batch size = 192 would yield slightly better results"

In [165]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
